In [ ]:
import requests
from datetime import datetime
import json
import traceback
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import lit, current_timestamp
from dateutil import parser as date_parser


### **Setup**


In [ ]:
spark = SparkSession.builder.appName("RentCastAPI").getOrCreate() 
API_KEY = ""   
BASE_URL = "https://api.rentcast.io/v1/listings/rental/long-term"
BATCH_SIZE = 500
CITY = "Long Beach"
STATE = "CA"
RAW_TABLE = "rentcast_raw_data"
CONTROL_TABLE = "rental_api_control"
ALLOW_SCHEMA_MERGE_ON_WRITE = True

### **Schema Definition**


In [ ]:
COLUMNS = [
    "addressLine1", "addressLine2", "bathrooms", "bedrooms", "city",
    "county", "countyFips", "createdDate", "daysOnMarket", "formattedAddress",
    "history", "hoa", "id", "lastSeenDate", "latitude", "listedDate",
    "listingAgent", "listingOffice", "listingType", "longitude", "lotSize",
    "mlsName", "mlsNumber", "price", "propertyType", "removedDate",
    "squareFootage", "state", "stateFips", "status", "yearBuilt", "zipCode"
]

data_schema = StructType([
    StructField("addressLine1", StringType(), True),
    StructField("addressLine2", StringType(), True),
    StructField("bathrooms", DoubleType(), True),
    StructField("bedrooms", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("county", StringType(), True),
    StructField("countyFips", StringType(), True),
    StructField("createdDate", TimestampType(), True),
    StructField("daysOnMarket", IntegerType(), True),
    StructField("formattedAddress", StringType(), True),
    StructField("history", StringType(), True),
    StructField("hoa", StringType(), True),
    StructField("id", StringType(), True),
    StructField("lastSeenDate", TimestampType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("listedDate", TimestampType(), True),
    StructField("listingAgent", StringType(), True),
    StructField("listingOffice", StringType(), True),
    StructField("listingType", StringType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("lotSize", DoubleType(), True),
    StructField("mlsName", StringType(), True),
    StructField("mlsNumber", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("propertyType", StringType(), True),
    StructField("removedDate", StringType(), True),
    StructField("squareFootage", IntegerType(), True),
    StructField("state", StringType(), True),
    StructField("stateFips", StringType(), True),
    StructField("status", StringType(), True),
    StructField("yearBuilt", IntegerType(), True),
    StructField("zipCode", StringType(), True)
])

### **Functions**


In [ ]:

def get_last_successful_offset():
    """Get the last successful offset from control table."""
    try:
        if not spark.catalog.tableExists(CONTROL_TABLE):
            print("Control table missing — starting at offset 0")
            return 0

        control_df = spark.sql(f"""
            SELECT offset_value, rows_extracted, total_available
            FROM {CONTROL_TABLE}
            WHERE status = 'SUCCESS'
            ORDER BY execution_date DESC
            LIMIT 1
        """)

        if control_df.count() == 0:
            print("No previous run — starting at offset 0")
            return 0

        row = control_df.first()
        last_offset = row['offset_value']
        last_rows = row['rows_extracted']
        last_total = row['total_available']
        next_offset = (last_offset or 0) + (last_rows or 0)

        print("Last successful run:")
        print(f"  - last_offset = {last_offset}")
        print(f"  - rows_extracted = {last_rows}")
        print(f"  - total_available = {last_total}")
        print(f"  - next_offset = {next_offset}")

        if last_total is not None and next_offset >= last_total:
            print("End reached previously — resetting to 0")
            return 0

        return next_offset

    except Exception as e:
        print(f"Offset lookup failed: {e}")
        return 0


def get_properties(city: str, state: str, offset: int, limit: int = BATCH_SIZE):
    headers = {"X-Api-Key": API_KEY, "Accept": "application/json"}
    params = {
        "city": city,
        "state": state,
        "limit": limit,
        "offset": offset,
        "includeTotalCount": "true"
    }
    resp = requests.get(BASE_URL, params=params, headers=headers, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    total_count = resp.headers.get("X-Total-Count")
    return data, int(total_count) if total_count else None


def parse_timestamp(value):
    if value is None or value == '' or value == 'None':
        return None
    try:
        # Use dateutil.parser for flexible parsing
        dt = date_parser.parse(value)
        # Remove timezone info to make it naive UTC
        return dt.replace(tzinfo=None)
    except:
        return None


def parse_int(value):
    if value is None or value == '' or value == 'None':
        return None
    try:
        return int(float(value))  
    except:
        return None


def parse_float(value):
    if value is None or value == '' or value == 'None':
        return None
    try:
        return float(value)
    except:
        return None


def parse_string(value):
    if value is None or value == '' or value == 'None' or str(value).strip() in ['', 'nan', '<NA>']:
        return None
    return str(value)


def parse_fips(value):
    if value is None or value == '' or value == 'None':
        return None
    try:
        # Convert to string, remove '.0' suffix if present
        s = str(value)
        if s.endswith('.0'):
            s = s[:-2]
        if s in ['', 'nan', '<NA>']:
            return None
        return s
    except:
        return None


def transform_row(prop):
    flat_prop = {}
    for k, v in prop.items():
        if isinstance(v, (dict, list)):
            flat_prop[k] = json.dumps(v) if v else None
        else:
            flat_prop[k] = v
    
    row = []
    for col in COLUMNS:
        value = flat_prop.get(col)
        
        if col in ["createdDate", "lastSeenDate", "listedDate", "removedDate"]:
            row.append(parse_timestamp(value))
        elif col in ["bedrooms", "daysOnMarket", "squareFootage", "yearBuilt"]:
            row.append(parse_int(value))
        elif col in ["bathrooms", "latitude", "longitude", "lotSize", "price"]:
            row.append(parse_float(value))
        elif col in ["countyFips", "stateFips"]:
            row.append(parse_fips(value))
        else:  # String columns
            row.append(parse_string(value))
    
    return tuple(row)


### **Checking the Offset**


In [ ]:
current_offset = get_last_successful_offset()
print(f"Starting extraction at offset: {current_offset}")
run_id = int(datetime.now().strftime("%Y%m%d%H%M%S"))
execution_date = datetime.now()
status = "FAILED"
error_message = None
rows_extracted = 0
total_available = 0

### **Get the Data**


In [ ]:
try:
    print(f"Calling API with offset: {current_offset}, limit: {BATCH_SIZE}")
    props, total_available = get_properties(CITY, STATE, current_offset, BATCH_SIZE)
    rows_extracted = len(props)
    print(f"Retrieved {rows_extracted} properties; total_available={total_available}")

    if rows_extracted == 0:
        status = "SUCCESS"
    else:
        print(f"Transforming {rows_extracted} properties...")
        data_rows = [transform_row(prop) for prop in props]
        
        #print(f"Transformed {len(data_rows)} rows")
        #print(f"\nSample values from first row:")
        first_row = data_rows[0]
        #print(f"  addressLine1: {first_row[0]} (type: {type(first_row[0])})")
        #print(f"  bathrooms: {first_row[2]} (type: {type(first_row[2])})")
        #print(f"  bedrooms: {first_row[3]} (type: {type(first_row[3])})")
        #print(f"  countyFips: {first_row[6]} (type: {type(first_row[6])})")
        #print(f"  createdDate: {first_row[7]} (type: {type(first_row[7])})")
        #print(f"  latitude: {first_row[14]} (type: {type(first_row[14])})")
        
        print(f"\nCreating Spark DataFrame...")
        df = spark.createDataFrame(data_rows, schema=data_schema)

        df = (
            df.withColumn("extraction_date", current_timestamp())
              .withColumn("offset_used", lit(current_offset))
              .withColumn("run_id", lit(run_id))
        )

        final_cols = COLUMNS + ["extraction_date", "offset_used", "run_id"]
        df = df.select(*final_cols)

        #print("\nSpark DataFrame schema:")
        #df.printSchema()
        
        print(f"\nRow count: {df.count()}")

        writer = (
            df.write
              .mode("append")
              .format("delta")
              .option("mergeSchema", "true" if ALLOW_SCHEMA_MERGE_ON_WRITE else "false")
        )
        writer.saveAsTable(RAW_TABLE)

        print(f"Wrote {df.count()} rows to Delta table: {RAW_TABLE}")
        status = "SUCCESS"

except Exception as e:
    error_message = str(e)
    print(f"\nError: {error_message}")
    traceback.print_exc()
    status = "FAILED"

print("\nSummary:")
print(f"   Status       : {status}")
print(f"   Rows extracted: {rows_extracted}")
print(f"   Offset used  : {current_offset}")

### **Update the Control Table**


In [ ]:
control_row = [(
    run_id,
    execution_date,
    current_offset,
    int(rows_extracted or 0),
    int(total_available or 0),
    status,
    error_message
)]

control_schema = StructType([
    StructField("run_id", LongType(), False),
    StructField("execution_date", TimestampType(), False),
    StructField("offset_value", IntegerType(), False),
    StructField("rows_extracted", IntegerType(), False),
    StructField("total_available", IntegerType(), False),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True)
])

try:
    cdf = spark.createDataFrame(control_row, control_schema)
    cdf.write.mode("append").saveAsTable(CONTROL_TABLE)
    print("Control row written")

except Exception as e:
    print(f"Failed to write control row: {e}")
    traceback.print_exc()

try:
    display(spark.sql(sql_hist))  
except NameError:
    spark.sql(sql_hist).show(truncate=False)